# M2.0b Primer: From Poisson's Equation to a Linear System

Book pointer: chapter 6, sections 6.1 (Laplace's and Poisson's equations), 6.2 (boundary conditions), 6.3 (finite difference methods), 6.4 (implicit methods).

Second of two primers for module 2. The notebooks that follow call relaxation and sparse solvers as finished tools. This one builds them from nothing. It starts in one dimension with five grid points, where every equation can be written out and every iteration done with a pencil, then moves to two dimensions, where the only change is that each point has four neighbours instead of two. By the end you will have written the matrix, done Jacobi sweeps by hand, watched them converge, and seen why the answer is unique.

## 1. Warm-up

1. Combine $\mathbf{E} = -\nabla V$ and $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ into one equation for $V$.
2. What is a boundary value problem? What extra information does a differential equation need to have one solution?
3. Write the finite-difference approximation to $d^2V/dx^2$ at grid point $i$.
4. On a grid, what does "$V$ satisfies Laplace's equation" say about $V_i$ and its neighbours?
5. Why does the solution of Laplace's equation with fixed boundary values have no interior maximum?

## 2. The idea in one paragraph

Poisson's equation relates the second derivatives of the potential to the charge density. On a grid, a second derivative is a combination of three neighbouring values, so the equation at each grid point is one linear equation in the unknown potentials at that point and its neighbours. One equation per unknown point, plus known values on the boundary, is a square linear system $A\mathbf{V} = \mathbf{b}$. You can solve it directly, which is what a sparse solver does, or you can solve it by repeatedly replacing each value by what its own equation demands, which is relaxation. The two give the same answer because the system has exactly one solution, and that uniqueness is a property of Laplace's equation, not of the algorithm.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sps
import scipy.sparse.linalg as spla
import time

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True, linewidth=120)

### 3.1 From Gauss to Poisson, and what a boundary value problem is

Substitute $\mathbf{E} = -\nabla V$ into $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$:
$$\nabla\cdot(-\nabla V) = \rho/\epsilon_0 \quad\Longrightarrow\quad \nabla^2 V = -\frac{\rho}{\epsilon_0}.$$
This is Poisson's equation. Where there is no charge it becomes Laplace's equation, $\nabla^2V = 0$. Both are second-order partial differential equations, and a second-order equation on its own does not pin down a solution: $V = 0$, $V = x$, and $V = x^2 - y^2$ all satisfy Laplace's equation. What pins it down is information on the boundary of the region. Two kinds appear:

- Dirichlet: the value of $V$ is given on the boundary. Physically, a conductor held at a known potential, or a far-away boundary where $V \to 0$.
- Neumann: the normal derivative $\partial V/\partial n$ is given on the boundary. Physically, the normal field $E_n$, which by section 3.7 of the previous primer means a known surface charge.

A region plus Poisson's equation inside plus one condition on every piece of the boundary is a boundary value problem, and it has exactly one solution (section 3.6). The job of a solver is to find it.

**Worked example 1.** In one dimension between two parallel plates at $x = 0$ and $x = L$ held at $V = 0$ and $V = V_0$, with no charge in between: $d^2V/dx^2 = 0$, so $V = c_1x + c_2$. The boundary values give $c_2 = 0$ and $c_1 = V_0/L$: $V = V_0x/L$, and $E = -dV/dx = -V_0/L$, uniform. With a uniform charge density $\rho_0$ between the plates: $d^2V/dx^2 = -\rho_0/\epsilon_0$, so $V = -\rho_0x^2/2\epsilon_0 + c_1x + c_2$. With $V(0) = 0$ and $V(L) = V_0$: $c_2 = 0$ and $c_1 = V_0/L + \rho_0L/2\epsilon_0$. The potential bulges upward by a parabola on top of the straight line. These two exact solutions are the test cases for everything below.

### 3.2 One dimension on five points, by hand

Take $L = 1$, $V(0) = 0$, $V(1) = 1$, and a grid of five points $x_i = 0, 0.25, 0.5, 0.75, 1$, so $h = 0.25$. The values $V_0 = 0$ and $V_4 = 1$ are known; $V_1, V_2, V_3$ are the unknowns. The second derivative at an interior point is approximated by the three-point formula (primer 00c, section 3.5):
$$\frac{d^2V}{dx^2}\Big|_i \approx \frac{V_{i-1} - 2V_i + V_{i+1}}{h^2}.$$
So Poisson's equation at point $i$ becomes the linear equation
$$V_{i-1} - 2V_i + V_{i+1} = -\frac{h^2\rho_i}{\epsilon_0}.$$

**Worked example 2 (Laplace, $\rho = 0$).** The three equations:
$$i = 1:\ V_0 - 2V_1 + V_2 = 0 \ \Rightarrow\ -2V_1 + V_2 = 0,$$
$$i = 2:\ V_1 - 2V_2 + V_3 = 0,$$
$$i = 3:\ V_2 - 2V_3 + V_4 = 0 \ \Rightarrow\ V_2 - 2V_3 = -1.$$
Known boundary values have moved to the right-hand side. In matrix form:
$$\begin{pmatrix} -2 & 1 & 0 \\ 1 & -2 & 1 \\ 0 & 1 & -2 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ V_3 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ -1 \end{pmatrix}.$$
Solve by substitution: from the first, $V_2 = 2V_1$. The second gives $V_1 - 4V_1 + V_3 = 0$, so $V_3 = 3V_1$. The third: $2V_1 - 6V_1 = -1$, so $V_1 = 0.25$, then $V_2 = 0.5$, $V_3 = 0.75$. Exactly $V = x$: the finite-difference equations are satisfied exactly by a linear function, because the three-point formula is exact for polynomials up to degree 3.

**Worked example 3 (Poisson, $\rho_0/\epsilon_0 = 8$).** The right-hand side gains $-h^2\rho/\epsilon_0 = -0.0625\cdot8 = -0.5$ at every interior point: $\mathbf{b} = (-0.5, -0.5, -1.5)$. The exact solution from worked example 1 is $V = -4x^2 + 5x$, giving $V(0.25) = 1.0$, $V(0.5) = 1.5$, $V(0.75) = 1.5$. The code solves both systems and shows the finite-difference answer is exact here too, since the solution is a quadratic.

In [ ]:
A = np.array([[-2.0, 1, 0], [1, -2, 1], [0, 1, -2]])
b_laplace = np.array([0.0, 0, -1])
b_poisson = np.array([-0.5, -0.5, -1.5])
print("Laplace, three unknowns:", np.linalg.solve(A, b_laplace), "   exact V = x at 0.25, 0.5, 0.75: [0.25 0.5 0.75]")
print("Poisson, rho/eps0 = 8:  ", np.linalg.solve(A, b_poisson), "   exact V = -4x^2 + 5x:           [1.   1.5  1.5 ]")

### 3.3 The same thing for any number of points

With $N$ interior points the matrix is $N\times N$ with $-2$ on the diagonal and $1$ on both neighbouring diagonals (tridiagonal). The right-hand side is $-h^2\rho_i/\epsilon_0$, with the two boundary values subtracted from the first and last entries. Build it, solve it, and compare to the exact parabola as the grid is refined. Because the solution is a quadratic the error is at rounding level for every $N$; for a smooth non-polynomial solution it would fall as $h^2$.

In [ ]:
def poisson_1d_matrix(N, h, rho_over_eps0, V_left, V_right):
    """Tridiagonal system for N interior points. Returns A (N x N) and b (N,)."""
    A = -2 * np.eye(N) + np.eye(N, k=1) + np.eye(N, k=-1)
    b = -h**2 * rho_over_eps0 * np.ones(N)
    b[0] -= V_left; b[-1] -= V_right
    return A, b

for N in (3, 7, 15, 63):
    h = 1.0 / (N + 1)
    xi = h * np.arange(1, N + 1)
    A, b = poisson_1d_matrix(N, h, 8.0, 0.0, 1.0)
    V = np.linalg.solve(A, b)
    print(f"N = {N:3d} interior points, h = {h:.4f}:  max |V - exact| = {np.abs(V - (-4 * xi**2 + 5 * xi)).max():.1e}")
print("\nthe 7-point matrix:"); print(poisson_1d_matrix(7, 0.125, 8.0, 0.0, 1.0)[0])

### 3.4 Relaxation: solving by averaging

Rearrange the equation at point $i$ to isolate $V_i$:
$$V_i = \frac{V_{i-1} + V_{i+1}}{2} + \frac{h^2\rho_i}{2\epsilon_0}.$$
Each point should equal the average of its neighbours, plus a source term. Jacobi's method takes that literally: start from any guess, compute a new value at every point from the old neighbours, repeat. Gauss-Seidel does the same but uses each new value as soon as it exists, sweeping left to right, which roughly halves the number of sweeps. Both stop when the largest change in a sweep drops below a tolerance.

**Worked example 4 (two Jacobi sweeps by hand).** Laplace on the five-point grid, start from $V = (0, 0, 0, 0, 1)$ (boundary values in place, interior zero).

Sweep 1, every interior point from the old neighbours: $V_1 = (0 + 0)/2 = 0$, $V_2 = (0 + 0)/2 = 0$, $V_3 = (0 + 1)/2 = 0.5$. Result $(0, 0, 0, 0.5, 1)$.

Sweep 2: $V_1 = (0 + 0)/2 = 0$, $V_2 = (0 + 0.5)/2 = 0.25$, $V_3 = (0 + 1)/2 = 0.5$. Result $(0, 0, 0.25, 0.5, 1)$.

Sweep 3: $V_1 = 0.125$, $V_2 = 0.25$, $V_3 = 0.625$. The boundary information creeps inward one point per sweep and the values climb toward $(0.25, 0.5, 0.75)$. That creeping is why the number of sweeps grows with the grid size: information has to cross the whole grid, and on a 2D grid of $N\times N$ points plain Jacobi needs of order $N^2$ sweeps.

**Worked example 5 (one Gauss-Seidel sweep by hand).** Same start. $V_1 = (0 + 0)/2 = 0$. $V_2 = (V_1^{\text{new}} + 0)/2 = 0$. $V_3 = (V_2^{\text{new}} + 1)/2 = 0.5$. Sweep 2: $V_1 = 0$, $V_2 = (0 + 0.5)/2 = 0.25$, $V_3 = (0.25 + 1)/2 = 0.625$. Already ahead of Jacobi at $V_3$, because it used the fresh $V_2$.

In [ ]:
def jacobi_sweep(V, src):
    Vn = V.copy()
    Vn[1:-1] = 0.5 * (V[:-2] + V[2:]) + 0.5 * src[1:-1]
    return Vn, np.abs(Vn - V).max()

def gauss_seidel_sweep(V, src):
    change = 0.0
    for i in range(1, len(V) - 1):
        new = 0.5 * (V[i - 1] + V[i + 1]) + 0.5 * src[i]
        change = max(change, abs(new - V[i])); V[i] = new
    return V, change

V = np.array([0.0, 0, 0, 0, 1.0]); src = np.zeros(5)
print("Jacobi by hand, sweep by sweep:")
for k in range(1, 4):
    V, _ = jacobi_sweep(V, src); print(f"  after sweep {k}: {V}")
V = np.array([0.0, 0, 0, 0, 1.0])
print("Gauss-Seidel:")
for k in range(1, 3):
    V, _ = gauss_seidel_sweep(V, src); print(f"  after sweep {k}: {V}")

# Convergence to tolerance on a larger grid, both methods.
N = 50; src = np.zeros(N + 2)
for name, sweep in (("Jacobi", jacobi_sweep), ("Gauss-Seidel", gauss_seidel_sweep)):
    V = np.zeros(N + 2); V[-1] = 1.0; history = []
    for k in range(100000):
        V, change = sweep(V, src); history.append(change)
        if change < 1e-8: break
    plt.semilogy(history, label=f"{name}: {len(history)} sweeps")
plt.xlabel("sweep"); plt.ylabel("largest change"); plt.legend(); plt.title("relaxation on 50 interior points"); plt.show()

Why does averaging converge to the right answer? Write $V = V_{\text{exact}} + e$. The exact solution satisfies the averaging equation, so the error $e$ satisfies the pure Laplace version: each sweep replaces $e_i$ by the average of its neighbours, with $e = 0$ on the boundary. Averaging never increases the largest value and smooths every bump, and the only function that equals the average of its neighbours everywhere and vanishes on the boundary is zero (that is the uniqueness argument of section 3.6). So $e \to 0$. Slowly, because long smooth bumps are barely changed by one averaging, which is also what over-relaxation (SOR, used in M2.2) speeds up.

### 3.5 Two dimensions

Nothing new happens except that each point has four neighbours. The five-point stencil (primer 00c) gives
$$V_{i-1,j} + V_{i+1,j} + V_{i,j-1} + V_{i,j+1} - 4V_{i,j} = -\frac{h^2\rho_{i,j}}{\epsilon_0},$$
one equation per interior node. To write a matrix we need to number the unknowns: the usual choice is row by row, $k = i\cdot n_y + j$, so that the neighbour $(i, j \pm 1)$ sits at $k \pm 1$ and the neighbour $(i \pm 1, j)$ at $k \pm n_y$. The matrix then has $-4$ on the diagonal and four off-diagonals of $1$, at distances $1$ and $n_y$. The cell prints it for a $3\times3$ block of unknowns.

**Worked example 6.** A $3\times3$ interior ($n = 5$ points per side, $h = 0.25$), boundary at $V = 0$ except the top edge ($j = 4$) at $V = 1$. Node $(2, 3)$, the middle of the top interior row, has neighbours $(1, 3)$, $(3, 3)$, $(2, 2)$ and the boundary node $(2, 4) = 1$. Its equation: $V_{1,3} + V_{3,3} + V_{2,2} - 4V_{2,3} = -1$. The known $1$ went to the right-hand side with a minus sign. The centre node $(2, 2)$ has four unknown neighbours and right-hand side $0$. By symmetry, the solution has $V_{1,j} = V_{3,j}$, and the code confirms the whole $3\times3$ answer.

In [ ]:
def poisson_2d_dense(n, V_boundary, rho_over_eps0=None):
    """Dense A V = b for an n x n grid with the outer ring fixed to V_boundary. Returns A, b, and an index map."""
    h = 1.0 / (n - 1)
    interior = [(i, j) for i in range(1, n - 1) for j in range(1, n - 1)]
    index = {ij: k for k, ij in enumerate(interior)}
    m = len(interior)
    A = np.zeros((m, m)); b = np.zeros(m)
    for (i, j), k in index.items():
        A[k, k] = -4.0
        if rho_over_eps0 is not None: b[k] = -h**2 * rho_over_eps0[i, j]
        for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
            if (ni, nj) in index: A[k, index[(ni, nj)]] = 1.0
            else: b[k] -= V_boundary[ni, nj]
    return A, b, index

n = 5
Vb = np.zeros((n, n)); Vb[:, -1] = 1.0                      # top edge (j = n-1) at 1 V
A, b, index = poisson_2d_dense(n, Vb)
print("A for the 3x3 interior (unknowns numbered row by row):"); print(A)
print("b:", b)
sol = np.linalg.solve(A, b)
V = Vb.copy()
for (i, j), k in index.items(): V[i, j] = sol[k]
print("\nV on the 5x5 grid (rows are i = x index, columns are j = y index):"); print(V)

### 3.6 Fixed nodes with a boolean mask, and the sparse matrix

Two practical tools that M2.2 uses without comment.

Masks. Conductors inside the region are nodes whose value is known. The clean way to say "these nodes are fixed" is a boolean array `fixed` of the same shape as `V`, `True` where the value is given. `V[~fixed]` selects the unknowns, `fixed[a:b, a:b] = True` marks a square block as a conductor, and in a relaxation sweep you simply skip the fixed nodes.

Sparse matrices. For an $n\times n$ grid there are about $n^2$ unknowns and the dense matrix has $n^4$ entries. At $n = 200$ that is $1.6\times10^9$ numbers, $13\,$GB, nearly all zeros. Each row has at most five non-zeros. SciPy's sparse formats store only those: build three lists (row index, column index, value), hand them to `scipy.sparse.csr_matrix`, and solve with `spsolve`. The assembly is the same loop as the dense version, just appending to lists instead of writing into a big array; the vectorised version in `emlib.relax.solve_sparse` does the same thing without the Python loop.

The cell solves the same problem with a conductor block inside three ways (Jacobi with a mask, dense, sparse) and shows that the sparse assembly is the only one that scales.

In [ ]:
def build_problem(n):
    V0 = np.zeros((n, n)); fixed = np.zeros((n, n), dtype=bool)
    V0[:, -1] = 1.0
    fixed[0, :] = fixed[-1, :] = fixed[:, 0] = fixed[:, -1] = True
    a, b_ = int(0.35 * n), int(0.55 * n)
    fixed[a:b_, a:b_] = True                                   # a grounded block: V0 is already 0 there
    return V0, fixed

def jacobi_2d(V0, fixed, tol=1e-6, max_sweeps=200000):
    V = V0.copy()
    for k in range(max_sweeps):
        Vn = V.copy()
        Vn[1:-1, 1:-1] = 0.25 * (V[2:, 1:-1] + V[:-2, 1:-1] + V[1:-1, 2:] + V[1:-1, :-2])
        Vn[fixed] = V0[fixed]                                  # fixed nodes keep their values
        change = np.abs(Vn - V).max(); V = Vn
        if change < tol: return V, k + 1
    return V, max_sweeps

def sparse_2d(V0, fixed):
    n = V0.shape[0]; idx = np.arange(n * n).reshape(n, n)
    rows, cols, vals = [], [], []; b = np.zeros(n * n)
    for i in range(n):
        for j in range(n):
            k = idx[i, j]
            if fixed[i, j]:
                rows.append(k); cols.append(k); vals.append(1.0); b[k] = V0[i, j]
            else:
                rows.append(k); cols.append(k); vals.append(-4.0)
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    rows.append(k); cols.append(idx[ni, nj]); vals.append(1.0)
    A = sps.csr_matrix((vals, (rows, cols)), shape=(n * n, n * n))
    return spla.spsolve(A, b).reshape(n, n)

n = 41
V0, fixed = build_problem(n)
t = time.time(); V_j, sweeps = jacobi_2d(V0, fixed, tol=1e-11); t_j = time.time() - t
t = time.time(); V_s = sparse_2d(V0, fixed); t_s = time.time() - t
print(f"n = {n}: Jacobi {sweeps} sweeps in {t_j:.2f} s, sparse direct in {t_s:.3f} s, max |difference| = {np.abs(V_j - V_s).max():.1e}")
print(f"dense matrix would have {(n * n) ** 2:,} entries; the sparse one stores {5 * (~fixed).sum() + fixed.sum():,}")

x = np.linspace(0, 1, n); X, Y = np.meshgrid(x, x, indexing="ij")
plt.contourf(X, Y, V_s, levels=21, cmap="viridis"); plt.colorbar(label="V")
plt.contour(X, Y, V_s, levels=11, colors="w", linewidths=0.5); plt.gca().set_aspect("equal"); plt.title("top edge at 1 V, grounded box and block"); plt.show()

### 3.7 Uniqueness, and why it matters

Suppose $V_1$ and $V_2$ both satisfy Poisson's equation in a region with the same values on the boundary. Their difference $W = V_1 - V_2$ satisfies Laplace's equation (the sources cancel) and is zero on the boundary. A solution of Laplace's equation equals the average of its neighbours everywhere, so it cannot have a maximum or minimum inside the region: an interior maximum would be larger than its neighbours, which contradicts being their average. So $W$ attains its largest and smallest values on the boundary, where it is zero. Hence $W = 0$ everywhere and $V_1 = V_2$.

Consequences you will use.

- Any method that produces a function satisfying the equation and the boundary conditions has produced the solution. Jacobi, Gauss-Seidel, the sparse solver, a lucky guess: all the same answer. The code checks this by starting Jacobi from a random guess.
- In M2.3, the method of images replaces a conductor by a fictitious charge. The resulting potential satisfies Laplace's equation outside the conductor and the right boundary values on it, so by uniqueness it is the solution there, however strange the construction looks.
- The field inside a closed conductor with no charge in its cavity is zero: $V$ is constant on the cavity wall, the constant satisfies Laplace's equation with those boundary values, so it is the solution.

In [ ]:
rng = np.random.default_rng(0)
V_random_start = V0.copy(); V_random_start[~fixed] = rng.uniform(-5, 5, size=(~fixed).sum())
V_r, sweeps_r = jacobi_2d(V_random_start, fixed, tol=1e-11)
V_z, sweeps_z = jacobi_2d(V0, fixed, tol=1e-11)
print(f"from a random guess in [-5, 5]: {sweeps_r} sweeps; from zero: {sweeps_z} sweeps; max |difference| = {np.abs(V_r - V_z).max():.1e}")
print(f"interior maximum of the Laplace solution: {V_s[~fixed].max():.4f}  <=  boundary maximum {V_s[fixed].max():.4f}   (no interior extremum)")

### 3.8 A Neumann condition

A given normal derivative is handled with a ghost point. At the left boundary $x_0$, require $dV/dx = g$. Place an imaginary point $x_{-1}$ and use the central difference $(V_1 - V_{-1})/2h = g$, so $V_{-1} = V_1 - 2hg$. Then the Poisson equation at $x_0$ itself becomes an equation for the unknown $V_0$: $V_{-1} - 2V_0 + V_1 = -h^2\rho_0/\epsilon_0$, that is $2V_1 - 2V_0 = -h^2\rho_0/\epsilon_0 + 2hg$. The boundary node has joined the unknowns, with a modified row.

**Worked example 7.** Laplace on $[0, 1]$ with $V'(0) = 2$ and $V(1) = 1$. The exact solution is the line with slope 2 through $(1, 1)$: $V = 2x - 1$. The code assembles the modified first row and recovers it.

In [ ]:
N_total = 9                                                 # nodes 0..8, node 8 fixed at V = 1, node 0 has a Neumann condition
h = 1.0 / (N_total - 1); g = 2.0
m = N_total - 1                                              # unknowns: nodes 0..7
A = -2 * np.eye(m) + np.eye(m, k=1) + np.eye(m, k=-1)
A[0, 1] = 2.0                                                # ghost-point row: 2 V_1 - 2 V_0
b = np.zeros(m); b[0] = 2 * h * g; b[-1] -= 1.0
V = np.linalg.solve(A, b)
xs = h * np.arange(m)
print("solver:", V); print("exact 2x - 1:", 2 * xs - 1)

### Exercise: assemble the 1D system yourself

Replace the body of `my_poisson_1d` so that it builds and solves the tridiagonal system for $N$ interior points with Dirichlet values at both ends and a given $\rho/\epsilon_0$ array. The check uses a non-uniform source $\rho/\epsilon_0 = \pi^2\sin(\pi x)$ with $V(0) = V(1) = 0$, whose exact solution is $V = \sin(\pi x)$, and requires second-order convergence.

In [ ]:
def my_poisson_1d(N, rho_over_eps0_interior, V_left, V_right):
    """Return V at the N interior points of [0, 1]."""
    h = 1.0 / (N + 1)
    A, b = poisson_1d_matrix(N, h, 0.0, V_left, V_right)           # replace from here with your own assembly
    b = b - h**2 * rho_over_eps0_interior
    return np.linalg.solve(A, b)


errors = []
for N in (7, 15, 31, 63):
    xi = np.arange(1, N + 1) / (N + 1)
    V = my_poisson_1d(N, np.pi**2 * np.sin(np.pi * xi), 0.0, 0.0)
    errors.append(np.abs(V - np.sin(np.pi * xi)).max())
    print(f"N = {N:3d}: max error {errors[-1]:.2e}")
ratios = np.array(errors[:-1]) / np.array(errors[1:])
print("error ratio per halving of h:", ratios, " (4 means second order)")
assert np.all(ratios > 3.5)

## 4. Verify

A 2D Poisson problem with a known answer. $V = \sin(\pi x)\sin(\pi y)$ vanishes on the boundary of the unit square and satisfies $\nabla^2V = -2\pi^2V$. So with $\rho/\epsilon_0 = 2\pi^2\sin(\pi x)\sin(\pi y)$ and $V = 0$ on the boundary, the solver must return $\sin(\pi x)\sin(\pi y)$, with an error that falls as $h^2$.

In [ ]:
def sparse_2d_poisson(n, rho_over_eps0):
    h = 1.0 / (n - 1); idx = np.arange(n * n).reshape(n, n)
    rows, cols, vals = [], [], []; b = np.zeros(n * n)
    for i in range(n):
        for j in range(n):
            k = idx[i, j]
            if i in (0, n - 1) or j in (0, n - 1):
                rows.append(k); cols.append(k); vals.append(1.0)
            else:
                rows.append(k); cols.append(k); vals.append(-4.0); b[k] = -h**2 * rho_over_eps0[i, j]
                for ni, nj in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)):
                    rows.append(k); cols.append(idx[ni, nj]); vals.append(1.0)
    return spla.spsolve(sps.csr_matrix((vals, (rows, cols)), shape=(n * n, n * n)), b).reshape(n, n)

errs = []
for n in (11, 21, 41, 81):
    x = np.linspace(0, 1, n); X, Y = np.meshgrid(x, x, indexing="ij")
    exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    V = sparse_2d_poisson(n, 2 * np.pi**2 * exact)
    errs.append(np.abs(V - exact).max()); print(f"n = {n:2d}: max error {errs[-1]:.2e}")
print("error ratio per halving of h:", np.array(errs[:-1]) / np.array(errs[1:]), " (4 means second order)")
assert np.all(np.array(errs[:-1]) / np.array(errs[1:]) > 3.5)

## 5. Explore and rebuild

Explore. Jacobi on the $n\times n$ box needs of order $n^2$ sweeps. Measure the sweep count for $n = 11, 21, 41, 81$ and confirm the exponent. Then implement over-relaxation: replace the Gauss-Seidel update $V_i \leftarrow V_i^{\text{GS}}$ by $V_i \leftarrow V_i + \omega(V_i^{\text{GS}} - V_i)$ with $\omega$ between 1 and 2, and find the $\omega$ that minimises the sweep count for $n = 41$. Compare with the formula used by `emlib.relax.sor_omega`. The gain is the reason M2.2 reaches for SOR.

Rebuild. Close this notebook. On paper, write the three-point and five-point stencils, assemble the $3\times3$ matrix of worked example 2 and solve it, and do the three Jacobi sweeps of worked example 4. Then write the uniqueness argument in four sentences. In a blank editor, write the 1D tridiagonal assembly and the 2D sparse assembly from scratch and reproduce the second-order convergence tables.